# Bayesian Risk-MLOps — Exploratory Data Analysis
**Author: Fidel Mehra**

This notebook covers:
1. Raw OHLCV data inspection & quality checks
2. Return distribution analysis (normality, fat-tails, skewness)
3. Volatility clustering (GARCH-eye test, ACF of squared returns)
4. Feature correlation heatmap
5. Rolling VaR visualisation
6. Stationarity tests (ADF, KPSS)
7. Tail risk: extreme value analysis (GEV fit)


In [ ]:
import sys
sys.path.insert(0, '../src')

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from pathlib import Path

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

cfg = yaml.safe_load(open('../config/config.yaml'))
PRIMARY = cfg['data']['primary_ticker']
print(f'Primary ticker: {PRIMARY}')

## 1. Load Raw OHLCV Data

In [ ]:
raw_path = Path(cfg['data']['cache_dir'])
start, end = cfg['data']['start_date'], cfg['data']['end_date']
ohlcv = pd.read_parquet(raw_path / f'{PRIMARY}_{start}_{end}.parquet')
print(ohlcv.shape)
ohlcv.head()

In [ ]:
# Data quality check
print('Missing values:')
print(ohlcv.isnull().sum())
print('\nData types:')
print(ohlcv.dtypes)
print('\nDate range:', ohlcv.index.min(), '->', ohlcv.index.max())

## 2. Price & Volume Overview

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(ohlcv.index, ohlcv['close'], linewidth=0.8, color='royalblue')
axes[0].set_title(f'{PRIMARY} Close Price')
axes[0].set_ylabel('Price ($)')

log_ret = np.log(ohlcv['close'] / ohlcv['close'].shift(1)).dropna()
axes[1].plot(log_ret.index, log_ret, linewidth=0.5, color='darkorange', alpha=0.8)
axes[1].set_title('Daily Log Returns')
axes[1].set_ylabel('Log return')
axes[1].axhline(0, color='grey', linestyle='--', linewidth=0.6)

axes[2].bar(ohlcv.index, ohlcv['volume'], width=1, color='steelblue', alpha=0.6)
axes[2].set_title('Daily Volume')
axes[2].set_ylabel('Volume')

plt.tight_layout()
plt.show()

## 3. Return Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Histogram vs normal
ax = axes[0]
ax.hist(log_ret, bins=80, density=True, color='steelblue', alpha=0.7, label='Empirical')
x = np.linspace(log_ret.min(), log_ret.max(), 200)
ax.plot(x, stats.norm.pdf(x, log_ret.mean(), log_ret.std()), 'r-', lw=2, label='Normal fit')
ax.set_title('Return Distribution vs Normal')
ax.legend()

# QQ-plot
ax = axes[1]
(osm, osr), (slope, intercept, r) = stats.probplot(log_ret, dist='norm')
ax.plot(osm, osr, 'o', markersize=2, alpha=0.5, color='steelblue')
ax.plot(osm, slope*np.array(osm)+intercept, 'r-', lw=2)
ax.set_title('QQ-Plot vs Normal')
ax.set_xlabel('Theoretical quantiles')
ax.set_ylabel('Sample quantiles')

# Excess kurtosis & skewness
ax = axes[2]
ax.axis('off')
stats_text = (
    f"Mean:          {log_ret.mean():.5f}\n"
    f"Std Dev:       {log_ret.std():.5f}\n"
    f"Skewness:      {log_ret.skew():.4f}\n"
    f"Ex. Kurtosis:  {log_ret.kurtosis():.4f}\n"
    f"JB p-value:    {stats.jarque_bera(log_ret).pvalue:.2e}"
)
ax.text(0.1, 0.5, stats_text, fontsize=12, family='monospace', verticalalignment='center')
ax.set_title('Summary Statistics')

plt.tight_layout()
plt.show()

## 4. Volatility Clustering (ACF of Squared Returns)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(log_ret**2, lags=40, ax=axes[0], title='ACF of Squared Returns (volatility clustering)')
plot_pacf(log_ret**2, lags=40, ax=axes[1], title='PACF of Squared Returns')
plt.tight_layout()
plt.show()

## 5. Stationarity Tests

In [ ]:
# ADF Test (H0: unit root = non-stationary)
adf_result = adfuller(log_ret.dropna(), autolag='AIC')
print(f'ADF statistic : {adf_result[0]:.4f}')
print(f'ADF p-value   : {adf_result[1]:.4e}')
print(f'Critical values: {adf_result[4]}')
print()

# KPSS Test (H0: stationary)
kpss_result = kpss(log_ret.dropna(), regression='c', nlags='auto')
print(f'KPSS statistic : {kpss_result[0]:.4f}')
print(f'KPSS p-value   : {kpss_result[1]:.4f}')
print(f'Critical values: {kpss_result[3]}')

## 6. Feature Matrix Correlation Heatmap

In [ ]:
feat_path = Path(cfg['data']['processed_dir']) / f'{PRIMARY}_features.parquet'
features = pd.read_parquet(feat_path)
print(f'Feature matrix shape: {features.shape}')

corr = features.drop(columns=['close'], errors='ignore').corr()

plt.figure(figsize=(18, 14))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.3,
            annot=False, cbar_kws={'shrink': 0.6})
plt.title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Rolling Value-at-Risk

In [ ]:
window = 63  # ~3 months
rolling_var = log_ret.rolling(window).quantile(0.05)
rolling_cvar = log_ret.rolling(window).apply(
    lambda x: x[x <= x.quantile(0.05)].mean()
)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(log_ret.index, log_ret, alpha=0.4, linewidth=0.5, color='steelblue', label='Log return')
ax.plot(rolling_var.index, rolling_var, color='darkorange', linewidth=1.2, label=f'{window}d Rolling VaR (5%)')
ax.plot(rolling_cvar.index, rolling_cvar, color='crimson', linewidth=1.2, linestyle='--', label=f'{window}d Rolling CVaR (5%)')
ax.axhline(0, color='grey', linestyle=':', linewidth=0.7)
ax.set_title(f'{PRIMARY} — Rolling VaR and CVaR')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Conclusions

- **Fat tails confirmed**: Jarque-Bera rejects normality; excess kurtosis > 0.
- **Volatility clustering**: ACF of squared returns is highly significant at multiple lags.
- **Stationarity**: ADF rejects unit root for log returns (p << 0.01).
- **Feature correlations**: Many rolling volatility features are highly correlated; PCA or regularisation (already provided by Bayesian Ridge) is warranted.
- **VaR dynamics**: Tail risk is time-varying and regime-dependent — motivates the probabilistic prediction approach.
